# Experiment 1: efficiency as sample size increases

The correctly specified DGP is $x=u+as+v$ and $y=\beta x+u+\epsilon$. We use $a\in\{1,2,3\}$, five sample sizes $n\in\{200,600,1000,1400,1800\}$, and 1,000 replications per design point. The unadjusted OLS baseline regresses $y$ on $x$ with an intercept and omits the confounder $u$. The other methods use the original sample-estimated IV and SV projections.

The main figure compares ordinary MSE and inference across all five sample sizes. OLS has a fixed confounding bias, so its ordinary MSE should level off. Scaled-MSE and other numerical summaries remain available in the CSV files.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

root = Path.cwd()
if not (root / "results" / "efficiency" / "efficiency_replications.csv").exists():
    root = root / "unification-control-modelling"
result_dir = root / "results" / "efficiency"
FIGURE_DIR = root / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def save_figure(fig, stem):
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", dpi=300, bbox_inches="tight",
                pad_inches=0.22, facecolor="white")

raw = pd.read_csv(result_dir / "efficiency_replications.csv")
summary = pd.read_csv(result_dir / "efficiency_summary.csv")
assert set(summary["replications"]) == {1000}
methods = ["ols_unadjusted", "iv", "sv", "equal", "feasible_efficient"]
labels = {"ols_unadjusted": "Unadjusted OLS", "iv": "IV only", "sv": "SV only", "equal": "Equal GMM", "feasible_efficient": "Efficient GMM"}
palette = sns.color_palette("colorblind", 8)
colors = dict(zip(methods, (palette[i] for i in (7, 0, 1, 2, 4))))
markers = {"ols_unadjusted": "X", "iv": "o", "sv": "s", "equal": "^", "feasible_efficient": "D"}
a_values = [1.0, 2.0, 3.0]
a_names = ["1", "2", "3"]
n_values = [200, 600, 1000, 1400, 1800]
assert set(summary["n"]) == set(n_values)
assert set(methods).issubset(set(summary["method"]))
assert len(raw) == 3 * 5 * 4 * 6 * 1000
print(f"Loaded {len(raw):,} simulation estimates.")


Loaded 360,000 simulation estimates.


## MSE, Type I error, and local power across sample sizes

The three panels show ordinary MSE, Type I error, and local power separately at all five sample sizes. Each panel has its own method legend below the axis. OLS uses its unadjusted Wald test.


In [2]:
est = raw[raw["scenario"] == "estimation"].copy()
est["error"] = est["beta_hat"] - est["beta"]
est["squared_error"] = est["error"] ** 2
metrics = est.groupby(["a", "n", "method"], as_index=False).agg(
    replications=("error", "size"),
    bias=("error", "mean"),
    mse=("squared_error", "mean"),
)
metrics["bias_squared"] = metrics["bias"] ** 2
metrics["variance"] = metrics["mse"] - metrics["bias_squared"]
assert np.allclose(metrics["mse"], metrics["bias_squared"] + metrics["variance"])
metrics.to_csv(result_dir / "efficiency_decomposition.csv", index=False)
metrics.head()

inference = raw[raw["scenario"].isin(["size", "power_1", "power_2"])].groupby(
    ["scenario", "a", "n", "method"], as_index=False
).agg(
    replications=("reject_h0_beta_0", "size"),
    rejection_rate=("reject_h0_beta_0", "mean"),
    coverage=("covered", "mean"),
)
inference.to_csv(result_dir / "efficiency_inference_summary.csv", index=False)


In [3]:
a = 2.0
plot_metrics = metrics[np.isclose(metrics["a"], a)].copy()
plot_metrics["scaled_mse"] = plot_metrics["n"] * plot_metrics["mse"]
plot_inference = inference[np.isclose(inference["a"], a) &
                           inference["scenario"].isin(["size", "power_2"])].copy()

from matplotlib.patches import Patch

journal_style = {
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix", "pdf.fonttype": 42,
    "font.size": 20, "axes.linewidth": 0.7, "axes.edgecolor": "0.65",
    "axes.labelsize": 21, "axes.titlesize": 22,
    "xtick.labelsize": 18, "ytick.labelsize": 18,
    "legend.fontsize": 18, "legend.title_fontsize": 19,
}
with plt.rc_context(journal_style):
    fig, axes = plt.subplots(1, 3, figsize=(18.5, 7.2))
    for method in methods:
        frame = plot_metrics[plot_metrics["method"].eq(method)].sort_values("n")
        axes[0].plot(frame["n"], frame["mse"],
                     color=colors[method], marker=markers[method], linewidth=1.8,
                     markersize=6, markerfacecolor="white", markeredgewidth=1.2,
                     label=labels[method])
    x = np.arange(len(n_values))
    bar_width = 0.14
    for axis, scenario in ((axes[1], "size"), (axes[2], "power_2")):
        values = plot_inference[plot_inference["scenario"].eq(scenario)].pivot(
            index="n", columns="method", values="rejection_rate"
        ).reindex(index=n_values, columns=methods)
        for index, method in enumerate(methods):
            axis.bar(x + (index - 2) * bar_width, values[method], bar_width,
                     color=colors[method], edgecolor="none", linewidth=0)
    axes[0].set(title="(a) Mean squared error", ylabel="MSE", yscale="log")
    axes[1].set(title="(b) Type I error", ylabel="Rejection probability", ylim=(0, 1.04))
    axes[2].set(title=r"(c) Local power: $\beta=2/\sqrt{n}$",
                ylabel="Rejection probability", ylim=(0, 1.04))
    axes[1].axhline(0.05, color="0.45", linestyle=":", linewidth=0.8)
    for ax in axes:
        ax.set_xlabel(r"Sample size $n$")
        ax.grid(axis="y", color="0.88", linewidth=0.5)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(direction="out", length=3, width=0.7)
    axes[0].set_xticks(n_values)
    for ax in axes[1:]:
        ax.set_xticks(x, n_values)
    line_handles, line_labels = axes[0].get_legend_handles_labels()
    axes[0].legend(handles=line_handles, labels=line_labels, title="Method",
                   loc="upper center", bbox_to_anchor=(0.5, -0.17),
                   ncol=2, frameon=False, columnspacing=0.8, handlelength=1.7)
    method_handles = [
        Patch(facecolor=colors[method], edgecolor="none", linewidth=0,
              label=labels[method]) for method in methods
    ]
    for ax in axes[1:]:
        ax.legend(handles=method_handles, title="Method", loc="upper center",
                  bbox_to_anchor=(0.5, -0.17), ncol=2, frameon=False,
                  columnspacing=0.8, handlelength=1.3)
    fig.subplots_adjust(left=0.065, right=0.99, bottom=0.35, top=0.92, wspace=0.30)
    save_figure(fig, "efficiency_mse_comparison")
    display(fig)
    plt.close(fig)
display(plot_metrics.pivot(index="n", columns="method", values="mse")[methods].rename(columns=labels).round(5))
for scenario, heading in [("size", "Type I error"), ("power_2", "Local power")]:
    print(heading)
    display(plot_inference[plot_inference["scenario"].eq(scenario)].pivot(
        index="n", columns="method", values="rejection_rate"
    )[methods].rename(columns=labels).round(3))


<Figure size 1850x720 with 3 Axes>

method,Unadjusted OLS,IV only,SV only,Equal GMM,Efficient GMM
n,,,,,
200,0.02927,0.00502,0.00510,0.00326,0.00259
600,0.02802,0.00174,0.00178,0.00117,0.00087
1000,0.02833,0.00096,0.00102,0.00064,0.00049
1400,0.02774,0.00071,0.00070,0.00048,0.00035
1800,0.02789,0.00057,0.00054,0.00038,0.00027


Type I error


method,Unadjusted OLS,IV only,SV only,Equal GMM,Efficient GMM
n,,,,,
200,0.991,0.053,0.057,0.055,0.055
600,1.000,0.049,0.059,0.056,0.062
1000,1.000,0.046,0.062,0.044,0.049
1400,1.000,0.059,0.057,0.055,0.071
1800,1.000,0.045,0.049,0.047,0.048


Local power


method,Unadjusted OLS,IV only,SV only,Equal GMM,Efficient GMM
n,,,,,
200,1.0,0.527,0.542,0.683,0.801
600,1.0,0.523,0.525,0.700,0.812
1000,1.0,0.547,0.537,0.719,0.838
1400,1.0,0.521,0.536,0.661,0.803
1800,1.0,0.528,0.504,0.678,0.819


## Compact summaries for each value of $a$

The table compares feasible efficient GMM with whichever single-channel method has the smaller MSE at that same $(a,n)$. The “MSE gain” is relative to that best single channel. Squared bias, variance, and MSE are reported separately so the source of the gain is visible. Full results for every method and $n$ are saved in `results/efficiency/efficiency_decomposition.csv`.


In [4]:
story_rows = []
for a, a_name in zip(a_values, a_names):
    for n in n_values:
        part = metrics[np.isclose(metrics["a"], a) & (metrics["n"] == n)].set_index("method")
        best_single = min(["iv", "sv"], key=lambda method: part.loc[method, "mse"])
        efficient = part.loc["feasible_efficient"]
        story_rows.append({
            "a": a, "n": n, "best_single": labels[best_single],
            "best_single_mse": part.loc[best_single, "mse"],
            "efficient_bias": efficient["bias"],
            "efficient_bias_squared": efficient["bias_squared"],
            "efficient_variance": efficient["variance"],
            "efficient_mse": efficient["mse"],
            "mse_gain": 1 - efficient["mse"] / part.loc[best_single, "mse"],
        })
story = pd.DataFrame(story_rows)
story.to_csv(result_dir / "efficiency_story_summary.csv", index=False)
for a, a_name in zip(a_values, a_names):
    print(f"a = {a_name}")
    part = story[np.isclose(story["a"], a)].drop(columns="a").set_index("n")
    display(part.style.format({
        "best_single_mse": "{:.5f}", "efficient_bias": "{:+.5f}",
        "efficient_bias_squared": "{:.6f}", "efficient_variance": "{:.5f}",
        "efficient_mse": "{:.5f}", "mse_gain": "{:.1%}",
    }))


a = 1


,best_single,best_single_mse,efficient_bias,efficient_bias_squared,efficient_variance,efficient_mse,mse_gain
n,,,,,,,
200,SV only,0.00564,-0.00249,0.000006,0.00452,0.00452,19.8%
600,SV only,0.00158,-0.00010,0.000000,0.00127,0.00127,19.6%
1000,SV only,0.00100,+0.00105,0.000001,0.00080,0.00080,20.0%
1400,SV only,0.00070,+0.00060,0.000000,0.00055,0.00055,22.6%
1800,SV only,0.00051,+0.00060,0.000000,0.00043,0.00043,16.1%


a = 2


,best_single,best_single_mse,efficient_bias,efficient_bias_squared,efficient_variance,efficient_mse,mse_gain
n,,,,,,,
200,IV only,0.00502,+0.00140,0.000002,0.00258,0.00259,48.5%
600,IV only,0.00174,-0.00194,0.000004,0.00086,0.00087,50.1%
1000,IV only,0.00096,+0.00047,0.000000,0.00049,0.00049,49.4%
1400,SV only,0.00070,-0.00060,0.000000,0.00035,0.00035,50.1%
1800,SV only,0.00054,+0.00025,0.000000,0.00027,0.00027,49.3%


a = 3


,best_single,best_single_mse,efficient_bias,efficient_bias_squared,efficient_variance,efficient_mse,mse_gain
n,,,,,,,
200,IV only,0.00236,-0.00010,0.000000,0.00162,0.00162,31.3%
600,IV only,0.00075,+0.00052,0.000000,0.00053,0.00053,28.8%
1000,IV only,0.00048,+0.00104,0.000001,0.00031,0.00031,34.8%
1400,IV only,0.00032,+0.00040,0.000000,0.00022,0.00022,32.0%
1800,IV only,0.00025,-0.00030,0.000000,0.00018,0.00018,30.3%


## Compact numerical benchmark

At $n=1000$, the table compares empirical scaled MSE with the efficient asymptotic benchmark. The OLS row is empirical only because fixed confounding bias prevents a finite scaled-MSE limit. The figure above is the main presentation; this table is a compact numerical reference.


In [5]:
main_rows = []
for a in a_values:
    part = metrics[np.isclose(metrics["a"], a) & (metrics["n"] == 1000)].set_index("method")
    best_single = min(part.loc["iv", "mse"], part.loc["sv", "mse"])
    main_rows.append({
        "a": a,
        "ols_scaled_mse": 1000 * part.loc["ols_unadjusted", "mse"],
        "iv_scaled_mse": 1000 * part.loc["iv", "mse"],
        "sv_scaled_mse": 1000 * part.loc["sv", "mse"],
        "equal_scaled_mse": 1000 * part.loc["equal", "mse"],
        "efficient_scaled_mse": 1000 * part.loc["feasible_efficient", "mse"],
        "oracle_efficient_bound": 4 / (a**2 + 4),
        "mse_gain_vs_best_single": 1 - part.loc["feasible_efficient", "mse"] / best_single,
    })
main_table = pd.DataFrame(main_rows).set_index("a")
main_table.to_csv(result_dir / "efficiency_main_table.csv")
display(main_table.round(3))


,ols_scaled_mse,iv_scaled_mse,sv_scaled_mse,equal_scaled_mse,efficient_scaled_mse,oracle_efficient_bound,mse_gain_vs_best_single
a,,,,,,,
1.0,112.984,3.975,1.001,0.811,0.801,0.800,0.200
2.0,28.334,0.962,1.021,0.640,0.487,0.500,0.494
3.0,8.495,0.476,1.059,0.429,0.310,0.308,0.348


## Inference summaries

The inference panels above use all five sample sizes. Full rejection-rate and coverage results for every $a$ and sample size remain in `efficiency_inference_summary.csv`.


In [6]:
inference.head()


,scenario,a,n,method,replications,rejection_rate,coverage
0,power_1,1.0,200,equal,1000,0.224,0.937
1,power_1,1.0,200,feasible_efficient,1000,0.211,0.942
2,power_1,1.0,200,iv,1000,0.101,0.951
3,power_1,1.0,200,ols_unadjusted,1000,1.000,0.000
4,power_1,1.0,200,oracle_efficient,1000,0.224,0.937


The compact table below pairs Type I error and local power for efficient GMM with the best single channel from the MSE comparison. The full inference CSV also contains the $\beta=1/\sqrt{n}$ power scenario and all methods.


In [7]:
infer_rows = []
for _, item in story.iterrows():
    a, n = item["a"], item["n"]
    best_method = "iv" if item["best_single"] == "IV only" else "sv"
    part = inference[np.isclose(inference["a"], a) & (inference["n"] == n)]
    rates = part.pivot(index="method", columns="scenario", values="rejection_rate")
    infer_rows.append({
        "a": a, "n": n,
        "efficient_type_i": rates.loc["feasible_efficient", "size"],
        "efficient_power_1": rates.loc["feasible_efficient", "power_1"],
        "efficient_power_2": rates.loc["feasible_efficient", "power_2"],
        "best_single": item["best_single"],
        "best_single_power_2": rates.loc[best_method, "power_2"],
    })
infer_table = pd.DataFrame(infer_rows)
infer_table.to_csv(result_dir / "efficiency_inference_story_summary.csv", index=False)
for a, a_name in zip(a_values, a_names):
    print(f"a = {a_name}")
    display(infer_table[np.isclose(infer_table["a"], a)].drop(columns="a").set_index("n").style.format({
        "efficient_type_i": "{:.3f}", "efficient_power_1": "{:.3f}",
        "efficient_power_2": "{:.3f}", "best_single_power_2": "{:.3f}",
    }))


a = 1


,efficient_type_i,efficient_power_1,efficient_power_2,best_single,best_single_power_2
n,,,,,
200,0.056,0.211,0.602,SV only,0.506
600,0.060,0.224,0.620,SV only,0.528
1000,0.048,0.189,0.617,SV only,0.525
1400,0.050,0.215,0.605,SV only,0.518
1800,0.047,0.201,0.630,SV only,0.522


a = 2


,efficient_type_i,efficient_power_1,efficient_power_2,best_single,best_single_power_2
n,,,,,
200,0.055,0.281,0.801,IV only,0.527
600,0.062,0.323,0.812,IV only,0.523
1000,0.049,0.310,0.838,IV only,0.547
1400,0.071,0.281,0.803,SV only,0.536
1800,0.048,0.305,0.819,SV only,0.504


a = 3


,efficient_type_i,efficient_power_1,efficient_power_2,best_single,best_single_power_2
n,,,,,
200,0.047,0.473,0.931,IV only,0.812
600,0.064,0.464,0.949,IV only,0.828
1000,0.062,0.448,0.930,IV only,0.817
1400,0.067,0.457,0.945,IV only,0.845
1800,0.068,0.445,0.952,IV only,0.848


## Interpretation

Unadjusted OLS exposes the persistent confounding bias: its ordinary MSE levels off and its scaled MSE grows with $n$. For IV, SV, and combined GMM, ordinary MSE falls with $n$; scaled MSE fluctuates around the relevant asymptotic risk. The stronger single channel changes with $a$, while efficient GMM combines information from both channels. The theoretical efficient risk is $4/(a^2+4)$ and the equal-weight risk is $(a^6/4+1)/(a^4/4+1)^2$.
